# Day 09. Exercise 02
# Metrics

## 0. Imports

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib


## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [12]:

# Preprocessing
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df['dayofweek'] = pd.read_csv('../data/dayofweek.csv')['dayofweek']

# Проверка целостности данных
assert df.shape[1] == 44, f'Expected 44 columns, got {df.shape[1]}'
assert len(df) == 1686, f'Expected 1686 rows, got {len(df)}'

# Разделение на признаки и целевую переменную
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

# Разбиение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

# Проверка распределения классов
print('Train distribution:')
print(y_train.value_counts().sort_index())
print(f'\nTest distribution:')
print(y_test.value_counts().sort_index())

Train distribution:
0    109
1    219
2    119
3    316
4     83
5    217
6    285
Name: dayofweek, dtype: int64

Test distribution:
0    27
1    55
2    30
3    80
4    21
5    54
6    71
Name: dayofweek, dtype: int64


## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.
2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [13]:
# Лучшие параметры SVM из предыдущего упражнения
best_params = {
    'C': 10,
    'kernel': 'rbf',
    'gamma': 'auto',
    'class_weight': None,
    'probability': True,
    'random_state': 21
}

# Обучение модели
model = SVC(**best_params)
model.fit(X_train, y_train)

# Предсказания
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# Метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# ROC AUC для мультиклассовой классификации
# multi_class='ovr' - one-vs-rest для всех классов
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

# Вывод результатов
print(f'accuracy is {accuracy:.5f}')
print(f'precision is {precision:.5f}')
print(f'recall is {recall:.5f}')
print(f'roc_auc is {roc_auc:.5f}')

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.98168


## 3. Decision tree

1. The same task for decision tree

In [14]:
# Лучшие параметры Decision Tree 
best_params = {
    'max_depth': 21,
    'class_weight': 'balanced',
    'criterion': 'gini',
    'random_state': 21
}

# Обучение модели
model = DecisionTreeClassifier(**best_params)
model.fit(X_train, y_train)

# Предсказания
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# Метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# ROC AUC для мультиклассовой классификации
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

# Вывод результатов
print(f'accuracy is {accuracy:.5f}')
print(f'precision is {precision:.5f}')
print(f'recall is {recall:.5f}')
print(f'roc_auc is {roc_auc:.5f}')

accuracy is 0.88462
precision is 0.88765
recall is 0.88462
roc_auc is 0.93638


## 4. Random forest

1. The same task for random forest.

In [15]:
# Лучшие параметры Random Forest из предыдущего упражнения
best_params = {
    'n_estimators': 100,
    'max_depth': 24,
    'class_weight': 'balanced',
    'criterion': 'entropy',
    'random_state': 21
}

# Обучение модели
model = RandomForestClassifier(**best_params)
model.fit(X_train, y_train)

# Предсказания
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

# Метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

# ROC AUC для мультиклассовой классификации
roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')

# Вывод результатов
print(f'accuracy is {accuracy:.5f}')
print(f'precision is {precision:.5f}')
print(f'recall is {recall:.5f}')
print(f'roc_auc is {roc_auc:.5f}')

accuracy is 0.92604
precision is 0.92754
recall is 0.92604
roc_auc is 0.99106


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [16]:
# 1. Выбор лучшей модели (случайный лес)
best_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=24,
    class_weight='balanced',
    criterion='entropy',
    random_state=21
)
best_model.fit(X_train, y_train)

# Предсказания на тесте
y_pred = best_model.predict(X_test)

# 2. Анализ ошибок

# --- Ошибки по weekday (целевая переменная) ---
# Считаем процент ошибок от ВСЕХ примеров этого класса в полном датасете
df_full = pd.read_csv('../data/dayofweek.csv')  # полный датасет с target
full_counts = df_full['dayofweek'].value_counts()

df_test = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred})
df_test['is_error'] = df_test['y_true'] != df_test['y_pred']
test_errors = df_test[df_test['is_error']].groupby('y_true').size()

error_rate_weekday = (test_errors / full_counts * 100).fillna(0)
print('Error rate by weekday (% of full dataset class):')
print(error_rate_weekday.sort_values(ascending=False).round(2))
print(f'Worst weekday: {error_rate_weekday.idxmax()} with {error_rate_weekday.max():.2f}% errors\n')

# --- Ошибки по labname ---
# Находим колонки labname в исходном датафрейме
lab_cols = [c for c in df.columns if c.startswith('labname_')]

# Для каждого misclassified примера смотрим, какие labname были активны (==1)
df_test_indexed = df_test.copy()
df_test_indexed.index = X_test.index  # восстанавливаем индексы

# Берём признаки для тестовой выборки
X_test_df = pd.DataFrame(X_test, columns=X.columns)

error_by_lab = {}
for lab in lab_cols:
    # Все примеры, где этот lab был активен
    lab_active = X_test_df[lab] == 1
    n_total = lab_active.sum()
    
    if n_total > 0:
        # Ошибки среди этих примеров
        n_errors = df_test_indexed[lab_active]['is_error'].sum()
        error_rate = n_errors / n_total * 100
        error_by_lab[lab] = error_rate

error_lab_df = pd.Series(error_by_lab).sort_values(ascending=False)
print('Error rate by labname (% of samples with that lab active):')
print(error_lab_df.head(10).round(2))
print(f'Worst labname: {error_lab_df.idxmax()} with {error_lab_df.max():.2f}% errors\n')

# --- Ошибки по users ---
user_cols = [c for c in df.columns if c.startswith('uid_user_')]

error_by_user = {}
for user in user_cols:
    user_active = X_test_df[user] == 1
    n_total = user_active.sum()
    
    if n_total > 0:
        n_errors = df_test_indexed[user_active]['is_error'].sum()
        error_rate = n_errors / n_total * 100
        error_by_user[user] = error_rate

error_user_df = pd.Series(error_by_user).sort_values(ascending=False)
print('Error rate by user (% of samples with that user active):')
print(error_user_df.head(10).round(2))
print(f'Worst user: {error_user_df.idxmax()} with {error_user_df.max():.2f}% errors\n')

# 3. Сохранение модели
joblib.dump(best_model, '../data/rf_metrics_model.pkl')
print('Model saved to ../data/rf_metrics_model.pkl')

Error rate by weekday (% of full dataset class):
0    4.41
4    2.88
5    1.85
1    1.46
2    1.34
3    0.76
6    0.56
dtype: float64
Worst weekday: 0 with 4.41% errors

Error rate by labname (% of samples with that lab active):
labname_lab03       100.00
labname_laba04       17.14
labname_lab05s       16.67
labname_laba06s      13.33
labname_laba06       11.11
labname_laba04s       8.00
labname_code_rvw      7.69
labname_project1      5.38
labname_laba05        2.13
labname_lab03s        0.00
dtype: float64
Worst labname: labname_lab03 with 100.00% errors

Error rate by user (% of samples with that user active):
uid_user_22    100.00
uid_user_6      50.00
uid_user_16     20.00
uid_user_27     16.67
uid_user_18     16.67
uid_user_3      14.29
uid_user_30     12.50
uid_user_31     11.11
uid_user_2      10.71
uid_user_19     10.53
dtype: float64
Worst user: uid_user_22 with 100.00% errors

Model saved to ../data/rf_metrics_model.pkl


In [17]:
# Проверим, сколько примеров с labname_lab03 было в тесте
lab = 'labname_lab03'
n_total = (X_test_df[lab] == 1).sum()
n_errors = df_test_indexed[(X_test_df[lab] == 1) & df_test_indexed['is_error']].shape[0]
print(f'{lab}: {n_errors} errors out of {n_total} samples in test')

labname_lab03: 1 errors out of 1 samples in test


In [18]:
# Проверка для uid_user_22
user = 'uid_user_22'
n_total = (X_test_df[user] == 1).sum()
n_errors = df_test_indexed[(X_test_df[user] == 1) & df_test_indexed['is_error']].shape[0]
print(f'{user}: {n_errors} errors out of {n_total} samples in test')

uid_user_22: 1 errors out of 1 samples in test


## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [19]:
def evaluate_models(models_list, params_list, X_train, y_train, X_test, y_test):
    """
    Оценивает несколько моделей и возвращает метрики для каждой.
    
    Parameters:
    -----------
    models_list : list
        Список классов моделей (например, [SVC, DecisionTreeClassifier, RandomForestClassifier])
    params_list : list of dict
        Список словарей с параметрами для каждой модели
    X_train, y_train : array-like
        Обучающая выборка
    X_test, y_test : array-like
        Тестовая выборка
    
    Returns:
    --------
    dict
        Словарь с метриками для каждой модели
    """
    results = {}
    
    for model_class, params in zip(models_list, params_list):
        # Создаём и обучаем модель
        model = model_class(**params)
        model.fit(X_train, y_train)
        
        # Предсказания
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)
        
        # Метрики
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
        
        # Сохраняем результаты
        model_name = model_class.__name__
        results[model_name] = {
            'accuracy': round(accuracy, 5),
            'precision': round(precision, 5),
            'recall': round(recall, 5),
            'roc_auc': round(roc_auc, 5)
        }
    
    return results

In [20]:
# Список моделей
models = [SVC, DecisionTreeClassifier, RandomForestClassifier]

# Список параметров для каждой модели
params = [
    {'C': 10, 'kernel': 'rbf', 'gamma': 'auto', 'class_weight': None, 
     'probability': True, 'random_state': 21},
    {'max_depth': 21, 'class_weight': 'balanced', 'criterion': 'gini', 
     'random_state': 21},
    {'n_estimators': 100, 'max_depth': 24, 'class_weight': 'balanced', 
     'criterion': 'entropy', 'random_state': 21}
]

# Вызов функции
metrics_dict = evaluate_models(models, params, X_train, y_train, X_test, y_test)

# Вывод результатов
for model_name, metrics in metrics_dict.items():
    print(f'\n{model_name}:')
    for metric, value in metrics.items():
        print(f'  {metric}: {value}')


SVC:
  accuracy: 0.88757
  precision: 0.89267
  recall: 0.88757
  roc_auc: 0.98168

DecisionTreeClassifier:
  accuracy: 0.88462
  precision: 0.88765
  recall: 0.88462
  roc_auc: 0.93638

RandomForestClassifier:
  accuracy: 0.92604
  precision: 0.92754
  recall: 0.92604
  roc_auc: 0.99106
